# 1.create tool and create llm 

In [24]:
from langchain_core.tools import tool

In [25]:
@tool
def multiply(a:int,b:int)->int:
    """this tool is used to multiply the two integers and give us integer in return"""
    return a*b

In [26]:
from dotenv import load_dotenv
load_dotenv()

True

In [27]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model = 'gpt-4o-mini')

In [28]:
llm.invoke('what is the capital of nepal')

AIMessage(content='The capital of Nepal is Kathmandu.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 14, 'total_tokens': 22, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'text_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None, 'video_tokens': 0}, 'cost': 6.9e-06, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 6.9e-06, 'upstream_inference_prompt_cost': 2.1e-06, 'upstream_inference_completions_cost': 4.8e-06}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-4o-mini', 'system_fingerprint': 'fp_369e662417', 'id': 'gen-1788506753-OJ2AHjQicKRkl8qlqqDo', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a06b4f-4917-7821-9be7-d722f26da458-0', tool_calls=[], invalid_tool_calls=[], usage

# 2.bind the llm with the tool

In [29]:
llm_with_tool = llm.bind_tools(tools=[multiply])

# 3. now invoke the llm with tool with the human message and see wheather the tool is called or not

In [30]:
from langchain_core.messages import HumanMessage

In [31]:
messages= [HumanMessage(content='what is 2 multiplied by 3')]

In [32]:
response = llm_with_tool.invoke(messages)

In [33]:
print(response.tool_calls)

[{'name': 'multiply', 'args': {'a': 2, 'b': 3}, 'id': 'call_CYcKOVgsgU9e7CX03GSkWzC4', 'type': 'tool_call'}]


In [34]:
messages.append(response)

In [35]:
messages

[HumanMessage(content='what is 2 multiplied by 3', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 64, 'total_tokens': 81, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'text_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None, 'video_tokens': 0}, 'cost': 1.98e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 1.98e-05, 'upstream_inference_prompt_cost': 9.6e-06, 'upstream_inference_completions_cost': 1.02e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-4o-mini', 'system_fingerprint': 'fp_af02af8afe', 'id': 'gen-1788506756-mreWwahBYBAdUeBA5T5p', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a06b4f-58

In [36]:
tool_result = multiply.invoke(response.tool_calls[0])

In [37]:
print(tool_result)

content='6' name='multiply' tool_call_id='call_CYcKOVgsgU9e7CX03GSkWzC4'


In [38]:
messages.append(tool_result)

In [39]:
messages 

[HumanMessage(content='what is 2 multiplied by 3', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 64, 'total_tokens': 81, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'text_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None, 'video_tokens': 0}, 'cost': 1.98e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 1.98e-05, 'upstream_inference_prompt_cost': 9.6e-06, 'upstream_inference_completions_cost': 1.02e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-4o-mini', 'system_fingerprint': 'fp_af02af8afe', 'id': 'gen-1788506756-mreWwahBYBAdUeBA5T5p', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a06b4f-58

In [41]:
llm_with_tool.invoke(messages).content

'2 multiplied by 3 is 6.'

## currency conversion tool

In [4]:
import requests
from langchain_core.tools import tool

In [5]:
@tool
def get_conversion_rate(base_currency:str , target_currency:str)->float:
    """this function fetches the conversion factor between the base_currency and the target currency"""
    url = f'https://v6.exchangerate-api.com/v6/c754eab14ffab33112e380ca/pair/{base_currency}/{target_currency}'

    response = requests.get(url = url)
    return response.json()

In [6]:
@tool
def conversion(base_currency_value:int,conversion_factor:float)->float:
     """given a currency conversion rate this function calculates the target currency value from a given base currency value"""
     return base_currency_value*conversion_factor

In [7]:
conversion.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'},
 'conversion_factor': {'title': 'Conversion Factor', 'type': 'number'}}

In [10]:
get_conversion_rate.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1788652801,
 'time_last_update_utc': 'Sun, 06 Sep 2026 00:00:01 +0000',
 'time_next_update_unix': 1788739201,
 'time_next_update_utc': 'Mon, 07 Sep 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 94.5116}

In [12]:
conversion.invoke({'base_currency_value':10,'conversion_factor':94.5116})

945.116

### tool binding

In [13]:
from langchain_openai import ChatOpenAI

In [14]:
llm = ChatOpenAI(model = 'gpt-4o-mini')

In [15]:
llm_with_tool= llm.bind_tools([get_conversion_rate,conversion])

In [16]:
from langchain_core.messages import HumanMessage

In [22]:
messages = []

In [23]:
messages = [HumanMessage('convert 10 usd into inr')]

In [24]:
response = llm_with_tool.invoke(messages)

In [25]:
response.tool_calls

[{'name': 'get_conversion_rate',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'call_VNuGMIHWCZMmqLJ95SO3nAeN',
  'type': 'tool_call'}]

### you may see that there is no call to the conversion tool 

In [26]:
messages.append(response)

In [27]:
messages

[HumanMessage(content='convert 10 usd into inr', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 106, 'total_tokens': 128, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'text_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None, 'video_tokens': 0}, 'cost': 2.91e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 2.91e-05, 'upstream_inference_prompt_cost': 1.59e-05, 'upstream_inference_completions_cost': 1.32e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-4o-mini', 'system_fingerprint': 'fp_9b48c9e51a', 'id': 'gen-1788670807-0NpSoMcETTG6kBCsQMbQ', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a07516-9

## now using the injected tool argument we can do that


In [28]:
# tool create
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/c754eab14ffab33112e380ca/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate


In [29]:
# tool binding
llm = ChatOpenAI()

In [30]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [31]:
messages = [HumanMessage('What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd')]

In [32]:
ai_message = llm_with_tools.invoke(messages)

In [33]:
messages.append(ai_message)

In [34]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'INR', 'target_currency': 'USD'},
  'id': 'call_BgF7hQS4tzS0esZgrRa2et7a',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10},
  'id': 'call_7Fzlrw7IQLjJ8BNpgoEQP2wQ',
  'type': 'tool_call'}]

## now you see that there are two tools

In [36]:
import json

In [37]:
import json

for tool_call in ai_message.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    messages.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    # fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)



In [38]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 123, 'total_tokens': 175, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'text_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None, 'video_tokens': 0}, 'cost': 0.0001395, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.0001395, 'upstream_inference_prompt_cost': 6.15e-05, 'upstream_inference_completions_cost': 7.8e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-3.5-turbo', 'system_fingerprint': None, 'id': 'gen-1788670981-au7knzI0ipZPCTdHRIVe', 'fini

In [39]:
llm_with_tools.invoke(messages).content

'The conversion factor between INR and USD is 0.01058. \n\nBased on this, the value of 10 INR is approximately 0.1058 USD.'